<a href="https://colab.research.google.com/github/MT-Maverick/NoteBooks/blob/main/Quarterly_Financials_for_SuperGroup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **OBJECTIVE: Identify financial metrics that contontribute to SuperGroup's EBITDA**

---


*   below we will go over the financials of SuperGroup and determine the main finacial metricts that have a correlation to it EBITDA for the past 4 quarters.
*   we will be using the data gatered from pythons libary yfinace which pulls data from yahoo finance.
*   This is purely for demonstartion purposes only, it is not finnacial advise  


In [ ]:
#Import libraries:
import pandas
import yfinance as yf
import numpy as np

#Set ticker for company we are interested in (Supergroup):
ticker = yf.Ticker('SGHC')

# **Overview of Data:**


---


below is a view of Supergroup's quarterly preformance data for the past 2 years and it's completness from our datasource

In [ ]:

# Fetch quarterly financial data
qf = ticker.quarterly_financials

# Fetch quarterly balance data:
qbs = ticker.quarterly_balance_sheet

# Display data completeness
print("Quarterly Financials:")
display(qf.info())

print("="*70)

# Display data completeness
print("Quarterly Balance Sheet:")
display(qbs.info())

# **Data Evaluation:**


---


Looking at the Index entries for the DataFrames, we see there is data missing for both finacial and balance sheet records, especially for the year 2025.this maybe because we are working with quarterly figures.

The objective is to evaluate the last 4 quarter's contributing factors to the EBITDA measure, so we can safely ignore the **2025-03 and 2025-06** data.

we will now proceed to set or features(X) and target(y) variables for modeling and graphing purposes.

In [ ]:
# get target variable for last 4 quarters:
y = qf.T['EBITDA'][:4]
display(y)


In [ ]:
# get features for quarterly financial data:
X = qf[qf.index != 'EBITDA'].T[:4]
display(X)

# **Data Imputation:**


---



Seeing that some of the features in our dateset range have None values: *(Total Unusual Items, Diluted Average Shares, and Basic EPS)*, we will fill the outstanding cells mean imputation since there is only one missing record for each column.

In [ ]:
# get features in quarterly financial data:
X = qf[qf.index != 'EBITDA'].T[:4]

# Fill missing values with averages for each financial metric (each column of transposed X)
X_filled = X.fillna(X.mean())

display(X_filled)

# **Determine the contributing features to EBITDA**


---
below we will deremine the correlation of **EBITDA** to other financial metrics in the quarterly financial statement


In [ ]:
# Calculate the correlation of each column in df against the target variable:
y_corr = X_filled.apply(lambda col: col.corr(y)).sort_values(ascending=False)
print(y_corr)




*   the **NaN** results we are getting at the bottom are because there are no captured records of: *(Gain On Sale Of Business,Earnings From Equity Interest,and Gain On Sale Of Security)*, thus when calculating the correlaton of EBITDA against them.

*   Looking at the correlation list, we can see, strong correlations belween a significant amount of features, however since some of these features are ajdustmets of EBITDA, we will focus on a select few.



In [ ]:
main_contributing_features = [
    'Tax Effect Of Unusual Items',
    'Total Unusual Items',
    'Reconciled Depreciation',
    'Diluted Average Shares',
    'Basic EPS',
    'Cost Of Revenue',
    'General And Administrative Expense',
    'Other Gand A',
    'Total Expenses',
    'Interest Expense',
    'Tax Rate For Calcs',
    'Minority Interests',
    'Interest Income'
]

mask = X_filled.columns.isin(main_contributing_features)
X_fs_filtered = X_filled.loc[:, mask]
X_fs_filtered

# **Data Visualisation:**


---


We will now display the selected features and their importance to the EBITDA according to the quarterly financials according to the decision tree model we will fit below.
  



In [ ]:
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt
import pandas as pd

# Initialize and fit a simple Decision Tree Regressor
# Using a small max_depth due to the limited number of samples (4 quarters)
model = DecisionTreeRegressor(max_depth=3, random_state=42)
model.fit(X_fs_filtered, y)

# Get feature importances
importances = model.feature_importances_
feature_impact = pd.Series(importances, index=X_fs_filtered.columns).sort_values(ascending=True)

# Plot the feature importances
plt.figure(figsize=(10, 6))
feature_impact.plot(kind='barh', color='skyblue')
plt.title('Impact of Each Feature on EBITDA (Decision Tree Feature Importance)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

Before viewing the balance sheet figures, let us reptresent the decision tree in graphical format.

In [ ]:
from sklearn.tree import plot_tree

# Plot decision tree structure in graphical form:
plt.figure(figsize=(12, 8))
plot_tree(
    model,
    feature_names=list(X_fs_filtered.columns),
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Structure for EBITDA Prediction (Filtered Features)", fontsize=14)
plt.show()

### Processing the Balance Sheet Data

In [ ]:
# Slice and transpose the last 4 quarters of balance sheet data
X_bs = qbs.T[:4]

# Impute missing values with column means
X_bs_filled = X_bs.fillna(X_bs.mean())

# Calculate correlations against EBITDA (y)
y_corr_bs = X_bs_filled.apply(lambda col: col.corr(y)).sort_values(ascending=False)
print("Top 20 Balance Sheet Correlations with EBITDA:")
print(y_corr_bs.head(20))

### Select and Filter Core Balance Sheet Features
We will select key representative financial health and asset metrics to avoid target leakage while retaining important structural elements.

In [ ]:
# Define key balance sheet features of interest
main_bs_features = [
    'Total Debt',
    'Tangible Book Value',
    'Cash And Cash Equivalents',
    'Ordinary Shares Number',
    'Working Capital',
    'Common Stock Equity',
    'Total Liabilities Net Minority Interest'
]

# Filter to available selected features
bs_mask = X_bs_filled.columns.isin(main_bs_features)
X_bs_filtered = X_bs_filled.loc[:, bs_mask]
display(X_bs_filtered)

### Train Decision Tree and Plot Feature Importances (Balance Sheet)

In [ ]:
# Train the Decision Tree Regressor on Balance Sheet features
model_bs = DecisionTreeRegressor(max_depth=3, random_state=42)
model_bs.fit(X_bs_filtered, y)

# Calculate and plot feature importances
importances_bs = model_bs.feature_importances_
feature_impact_bs = pd.Series(importances_bs, index=X_bs_filtered.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_impact_bs.plot(kind='barh', color='lightcoral')
plt.title('Impact of Balance Sheet Features on EBITDA (Feature Importance)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

### Visualize Balance Tree Structure

In [ ]:
plt.figure(figsize=(12, 8))
plot_tree(
    model_bs,
    feature_names=list(X_bs_filtered.columns),
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Structure for EBITDA Prediction (Balance Sheet)", fontsize=14)
plt.show()

### Combining Datasets and Training a Random Forest

In [ ]:
rf_impact.index

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import matplotlib.pyplot as plt

# Combine financials (using correct spelling) and balance sheet features
X_combined = pd.concat([X_fs_filtered, X_bs_filtered], axis=1)

# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
rf_model.fit(X_combined, y)

# Calculate and plot feature importances
rf_importances = rf_model.feature_importances_
rf_impact = pd.Series(rf_importances, index=X_combined.columns).sort_values(ascending=True)

# Plot
plt.figure(figsize=(12, 8))
rf_impact.plot(kind='barh', color='plum')
plt.title('Overall Main Contributing Factors to EBITDA (Random Forest - Corrected)', fontsize=14)
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

### Scenario Analysis & EBITDA Stress Testing
Below, we define a helper function that leverages our trained Random Forest model to predict how EBITDA changes if we adjust key operational and financial levers (e.g., decreasing Cash and Cash Equivalents, changing Cost of Revenue, or optimizing Working Capital).

In [ ]:
import matplotlib.pyplot as plt

def simulate_ebitda_change(
    cash_percent_change=0.0,
    cost_of_rev_percent_change=0.0,
    working_cap_percent_change=0.0,
    total_expenses_percent_change=0.0,
    total_debt_percent_change=0.0
):
    """
    Simulates the impact on EBITDA based on percentage modifications to key drivers
    using the last available quarter (2026-06-30) as the baseline and plots a comparison chart.
    """
    # Copy the baseline row (most recent quarter)
    baseline_scenario = X_combined.iloc[[0]].copy()

    # Apply modifications to selected features if present in the data
    if 'Cash And Cash Equivalents' in baseline_scenario.columns:
        baseline_scenario['Cash And Cash Equivalents'] *= (1.0 + cash_percent_change)
    if 'Cost Of Revenue' in baseline_scenario.columns:
        baseline_scenario['Cost Of Revenue'] *= (1.0 + cost_of_rev_percent_change)
    if 'Working Capital' in baseline_scenario.columns:
        baseline_scenario['Working Capital'] *= (1.0 + working_cap_percent_change)
    if 'Total Expenses' in baseline_scenario.columns:
        baseline_scenario['Total Expenses'] *= (1.0 + total_expenses_percent_change)
    if 'Total Debt' in baseline_scenario.columns:
        baseline_scenario['Total Debt'] *= (1.0 + total_debt_percent_change)

    # Predict new EBITDA
    simulated_pred = rf_model.predict(baseline_scenario)[0]
    baseline_pred = rf_model.predict(X_combined.iloc[[0]])[0]

    print(f"== EBITDA Simulation Results (Base Quarter 2026-06) ==")
    print(f"Baseline Predicted EBITDA (2026-06):  ${baseline_pred:,.2f}")
    print(f"Simulated Predicted EBITDA (2026-10): ${simulated_pred:,.2f}")
    print(f"Net Change:                 ${simulated_pred - baseline_pred:+,.2f} ({(simulated_pred - baseline_pred)/baseline_pred * 100:+.2f}%)")
    print("="*80)

    # Visual Representation (Histogram / Bar Comparison)
    plt.figure(figsize=(8, 5))
    categories = ['Baseline EBITDA (2026-06)', 'Simulated EBITDA (2026-10)']
    values = [baseline_pred / 1e6, simulated_pred / 1e6] # Convert to Millions for scaling
    colors = ['skyblue', 'plum']

    bars = plt.bar(categories, values, color=colors, width=0.5, edgecolor='grey')
    plt.title('EBITDA Simulation Comparison ($ Millions)', fontsize=14)
    plt.ylabel('EBITDA ($ Millions)', fontsize=12)

    # Add values on top of the bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"${yval:.2f}M", ha='center', va='bottom', fontweight='bold')

    plt.ylim(0, max(values) * 1.25)
    plt.tight_layout()
    plt.show()

# Example Scenario: What if Cash drops by 10% and Cost of Revenue increases by 5%?
simulate_ebitda_change(cash_percent_change=-0.10, cost_of_rev_percent_change=0.05)